In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!unzip /content/datasetnew9.zip

Archive:  /content/datasetnew9.zip
   creating: datasetnew/
  inflating: __MACOSX/._datasetnew   
  inflating: datasetnew/.DS_Store    
  inflating: __MACOSX/datasetnew/._.DS_Store  
   creating: datasetnew/bad/
  inflating: __MACOSX/datasetnew/._bad  
   creating: datasetnew/good/
  inflating: __MACOSX/datasetnew/._good  
  inflating: datasetnew/bad/40bed.json  
  inflating: __MACOSX/datasetnew/bad/._40bed.json  
  inflating: datasetnew/bad/33.bad.json  
  inflating: __MACOSX/datasetnew/bad/._33.bad.json  
  inflating: datasetnew/bad/32.bad.json  
  inflating: __MACOSX/datasetnew/bad/._32.bad.json  
  inflating: datasetnew/bad/22bed.json  
  inflating: __MACOSX/datasetnew/bad/._22bed.json  
  inflating: datasetnew/bad/23bed.json  
  inflating: __MACOSX/datasetnew/bad/._23bed.json  
  inflating: datasetnew/bad/8.bad.json  
  inflating: __MACOSX/datasetnew/bad/._8.bad.json  
  inflating: datasetnew/bad/9.bad.json  
  inflating: __MACOSX/datasetnew/bad/._9.bad.json  
  inflating: dataset

In [ ]:
import os
import json
from datasets import Dataset

# Paths for your dataset
good_dir = "/content/datasetnew/good"
bad_dir = "/content/datasetnew/bad"

# Function to load your JSON dataset (Good and Bad)
def load_good_bad_jsons(good_dir, bad_dir):
    data = []
    good_files = sorted(os.listdir(good_dir))
    bad_files = sorted(os.listdir(bad_dir))

    for good_file, bad_file in zip(good_files, bad_files):
        if good_file.endswith('.json') and bad_file.endswith('.json'):
            try:
                with open(os.path.join(good_dir, good_file), 'r', encoding='utf-8') as gf, \
                     open(os.path.join(bad_dir, bad_file), 'r', encoding='utf-8') as bf:
                    good_json = json.load(gf)
                    bad_json = json.load(bf)

                    data.append({
                        'input_text': json.dumps(bad_json),
                        'target_text': json.dumps(good_json)
                    })
            except json.JSONDecodeError as e:
                print(f"Skipping {good_file} or {bad_file} due to JSON error: {e}")
            except Exception as e:
                print(f"Skipping {good_file} or {bad_file} due to unexpected error: {e}")

    return Dataset.from_list(data)

# Load dataset
dataset = load_good_bad_jsons(good_dir, bad_dir)
dataset = dataset.train_test_split(test_size=0.1)

print(dataset)


Skipping 16.good.json or 16.bad.json due to JSON error: Unterminated string starting at: line 20 column 11 (char 776)
DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 36
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 5
    })
})


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model name
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name, load_in_4bit=True, device_map="auto")

print("Model and tokenizer loaded.")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model and tokenizer loaded.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the model for 4-bit training (reduces memory usage)
model = prepare_model_for_kbit_training(model)

# Apply LoRA configuration for efficient fine-tuning
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # fine-tune only projections to save memory
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

print("Model prepared for 4-bit training with LoRA.")


Model prepared for 4-bit training with LoRA.


In [ ]:
# Function to generate prompts
def generate_prompt(example):
    return f"""### Bad PPTX Content JSON:
{example['input_text']}

### Enhanced PPTX Content JSON:
{example['target_text']}"""

# Tokenize the dataset
def tokenize_function(examples):
    full_prompt = generate_prompt(examples)
    tokenized = tokenizer(
        full_prompt,
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    # For causal LM, labels = input_ids
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=["input_text", "target_text"],
    batched=False  # or True if you batch
)



Map:   0%|          | 0/36 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments as HFTrainingArguments

output_dir = "/content/enhanced-pptx-model"

training_args = HFTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    # remove evaluation_strategy—use eval_steps directly
    eval_steps=50,
    save_steps=50,

    num_train_epochs=10,
    learning_rate=2e-4,
    bf16=False,            # set True on TPU or BF16-capable GPU
    logging_steps=10,
    save_total_limit=2,

    push_to_hub=False,
    report_to="none",
)


In [ ]:
from transformers import Trainer, default_data_collator

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=default_data_collator,
)


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,3.650500
20,2.011000
30,1.649700
40,1.566200


TrainOutput(global_step=40, training_loss=2.2193341493606566, metrics={'train_runtime': 284.3471, 'train_samples_per_second': 1.266, 'train_steps_per_second': 0.141, 'total_flos': 916266915201024.0, 'train_loss': 2.2193341493606566, 'epoch': 8.0})

In [ ]:
# Save the fine-tuned model
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ Fine-tuning complete! Model saved at:", output_dir)


✅ Fine-tuning complete! Model saved at: /content/enhanced-pptx-model


In [ ]:
# 1. Install the Hub client (if you haven’t yet)
!pip install -q huggingface_hub

# 2. Login (this will prompt you to paste your HF token)
from huggingface_hub import notebook_login
notebook_login()

# 3. Save your locally fine-tuned model & tokenizer (if not already saved)
output_dir = "/content/enhanced-pptx-model"  # same as your TrainingArguments output_dir
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# 4. Push to your HF repo (replace with your username and desired repo name)
# Ensure that you have write access to this repo or create a new one under your username
hf_repo = "Faisalkhany/llam1b_finetune"  # Replace with your username and repo name
model.push_to_hub(hf_repo)
tokenizer.push_to_hub(hf_repo)

print(f"✅ Pushed to https://huggingface.co/{hf_repo}")

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Pushed to https://huggingface.co/Faisalkhany/llam1b_finetune


In [ ]:
!pip install -q streamlit python-pptx transformers pyngrok


In [ ]:
%%writefile app.py

Overwriting app.py


In [ ]:
import subprocess
import threading
import time

# Start streamlit server (your existing app.py)
def run_streamlit():
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true"])

thread = threading.Thread(target=run_streamlit)
thread.start()

# Wait a few seconds to make sure Streamlit starts
time.sleep(5)


In [ ]:
from pyngrok import ngrok
ngrok.kill()  # Kill old tunnels
ngrok.set_auth_token("2wNjYLD33sDLRKSESiDOmTMVOjn_BxHSRqt9hPMsqheoUZ2F")
public_url = ngrok.connect(8501)
print('🚀 Your app is live at:', public_url)


🚀 Your app is live at: NgrokTunnel: "https://ca26-34-16-170-185.ngrok-free.app" -> "http://localhost:8501"
